# 🦀 双摆（Double Pendulum）数值模拟 — Ver 1

## 项目背景

本 notebook 是 **PINN + NODE 双摆对比项目** 的第①步：

> **双摆运动方程推导 → 数值积分 → 轨迹生成 → 物理验证**

最终目标：生成用于 PINN（物理约束求解）和 NODE（数据驱动系统辨识）对比的标准轨迹数据集。

---

## 双摆是什么？

```
       固定点
         │
         θ₁    ← 第一连杆
         │
        m₁ ──── 第一个质点
         │
         θ₂    ← 第二连杆
         │
        m₂ ──── 第二个质点
```

**为什么用双摆？**
- 2 自由度（$"θ_1, θ_2$）：足够复杂展示耦合动力学
- **混沌运动**：对初始条件敏感，是测试模型是否"真懂"物理的绝佳 benchmark
- 无摩擦时**能量守恒**：可用来验证数值精度和模型质量
- 直接对应**双连杆机械臂**：去掉重力加力矩就是机器人动力学基础

---

## 本 notebook 包含的内容

| 模块 | 物理对应 | 内容 |
|:---|:--------|:----|
| 1️⃣ 参数定义 | $m_1, m_2, l_1, l_2, g$ | 系统物理参数和积分参数 |
| 2️⃣ ODE 方程 | $\frac{d\mathbf{s}}{dt} = f(\mathbf{s})$ | 从拉格朗日推导出的运动方程 |
| 3️⃣ 数值积分 | `solve_ivp` | 用 scipy 解 ODE 得到轨迹 |
| 4️⃣ 可视化 | $\theta(t), E(t)$ | 验证结果是否物理合理 |

---
## 1️⃣ 参数配置

定义系统的物理参数和数值积分设置。

**物理参数：**
- $m_1, m_2$：两个质点的质量 [kg]
- $l_1, l_2$：两段无质量硬杆的长度 [m]
- $g$：重力加速度 [m/s²]

**初始条件：**
- $\theta_{10}, \theta_{20}$：初始角度 [rad]
- $\omega_{10}, \omega_{20}$：初始角速度 [rad/s]

**数值参数：**
- `t_span`：积分时间区间
- `dt`：采样步长
- `rtol, atol`：误差容限（越小越精确）

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from pathlib import Path

# 设置随机种子，确保可复现
np.random.seed(42)

# ─── 物理参数 ───
M1 = 1.0      # 第一个质点的质量 [kg]
M2 = 1.0      # 第二个质点的质量 [kg]
L1 = 1.0      # 第一根杆长度 [m]
L2 = 1.0      # 第二根杆长度 [m]
G  = 9.81     # 重力加速度 [m/s²]

# ─── 默认初始条件 ───
# θ₁=π/2, θ₂=π/2：从水平位置释放，是最经典的混沌测试条件
THETA1_0 = np.pi / 2.0
THETA2_0 = np.pi / 2.0
OMEGA1_0 = 0.0
OMEGA2_0 = 0.0

# ─── 数值积分参数 ───
T_SPAN = [0.0, 20.0]   # 模拟 20 秒
DT     = 0.01           # 100 Hz 采样
METHOD = "RK45"        # 4(5)阶 Runge-Kutta
RTOL   = 1e-9           # 相对误差容限
ATOL   = 1e-12          # 绝对误差容限

# ─── 数据生成参数 ───
N_TRAJECTORIES = 50     # 批量生成轨迹数
THETA_RANGE    = [-np.pi, np.pi]
OMEGA_RANGE    = [-3.0, 3.0]

---
## 2️⃣ 双摆动力学方程

### 从拉格朗日量到运动方程

系统状态由 4 个量完整描述：

$$\mathbf{s} = [\theta_1, \theta_2, \omega_1, \omega_2]^T$$

其中 $\omega_i = \dot{\theta}_i$ 是角速度。

### ODE 右端函数

$$\frac{d\mathbf{s}}{dt} = \begin{bmatrix} \dot{\theta}_1 \\ \dot{\theta}_2 \\ \ddot{\theta}_1 \\ \ddot{\theta}_2 \end{bmatrix} = \begin{bmatrix} \omega_1 \\ \omega_2 \\ \alpha_1(\mathbf{s}) \\ \alpha_2(\mathbf{s}) \end{bmatrix}$$

### 角加速度 $\alpha = M^{-1}F$

耦合的二阶 ODE 整理为矩阵形式：

$$\begin{bmatrix} (m_1+m_2)l_1 & m_2 l_2 \cos(\theta_1-\theta_2) \\ m_2 l_1 \cos(\theta_1-\theta_2) & m_2 l_2 \end{bmatrix}
\begin{bmatrix} \ddot{\theta}_1 \\ \ddot{\theta}_2 \end{bmatrix} =
\begin{bmatrix} 
-m_2 l_2 \dot{\theta}_2^2 \sin(\theta_1-\theta_2) - (m_1+m_2)g \sin\theta_1 \\
m_2 l_1 \dot{\theta}_1^2 \sin(\theta_1-\theta_2) - m_2 g \sin\theta_2
\end{bmatrix}$$

### 能量表达式

动能 $T = \frac{1}{2}(m_1+m_2)l_1^2\omega_1^2 + \frac{1}{2}m_2l_2^2\omega_2^2 + m_2l_1l_2\omega_1\omega_2\cos(\theta_1-\theta_2)$

势能 $V = -(m_1+m_2)gl_1\cos\theta_1 - m_2gl_2\cos\theta_2$

总能量 $E = T + V$ 在无摩擦时守恒。

In [ ]:
def derivatives(t, state, m1, m2, l1, l2, g):
    """
    双摆 ODE 右端函数 f(s)。
    
    给入当前状态 s = [θ₁, θ₂, ω₁, ω₂]，
    返回 ds/dt = [ω₁, ω₂, α₁, α₂]。
    
    核心公式：
        dθ₁/dt = ω₁
        dθ₂/dt = ω₂
        M(θ₁,θ₂) · α = F(θ₁,θ₂,ω₁,ω₂)
        α = M⁻¹ · F
    """
    θ1, θ2, ω1, ω2 = state
    
    # 角度差在耦合项中反复出现，提前计算一次
    Δθ = θ1 - θ2
    cΔ = np.cos(Δθ)
    sΔ = np.sin(Δθ)
    
    # ── 质量矩阵 M (2×2) ──
    # M₁₁ = (m₁+m₂)l₁ : m₁+m₂ 整体转动
    # M₁₂ = m₂l₂·cos(θ₁-θ₂) : 耦合项
    # M₂₂ = m₂l₂ : m₂ 自身转动
    M11 = (m1 + m2) * l1
    M12 = m2 * l2 * cΔ
    M21 = m2 * l1 * cΔ
    M22 = m2 * l2
    
    # ── 右手边 F (2×1) ──
    # F₁ = 离心力 + 重力回复力
    # F₂ = 离心力反作用 + 重力回复力
    F1 = -m2 * l2 * ω2**2 * sΔ - (m1 + m2) * g * np.sin(θ1)
    F2 =  m2 * l1 * ω1**2 * sΔ - m2 * g * np.sin(θ2)
    
    # ── 求解 α = M⁻¹·F ──
    # 2×2 矩阵逆解析形式:
    #   M⁻¹ = 1/det · [[M₂₂, -M₁₂], [-M₂₁, M₁₁]]
    det = M11 * M22 - M12 * M21
    
    if abs(det) < 1e-12:
        raise ValueError(f"质量矩阵奇异! det={det:.2e}")
    
    α1 = (M22 * F1 - M12 * F2) / det
    α2 = (M11 * F2 - M21 * F1) / det
    
    return np.array([ω1, ω2, α1, α2])


def compute_energy(state, m1, m2, l1, l2, g):
    """
    计算总能量 E = T + V。
    
    能量守恒是验证数值积分精度的核心指标。
    """
    θ1, θ2, ω1, ω2 = state
    Δθ = θ1 - θ2
    
    # 动能
    T = (0.5 * (m1 + m2) * l1**2 * ω1**2
       + 0.5 * m2 * l2**2 * ω2**2
       + m2 * l1 * l2 * ω1 * ω2 * np.cos(Δθ))
    
    # 势能
    V = (-(m1 + m2) * g * l1 * np.cos(θ1)
         - m2 * g * l2 * np.cos(θ2))
    
    return T + V, T, V


def compute_positions(state, l1, l2):
    """
    从广义坐标 θ₁, θ₂ 计算质点的 (x, y) 坐标。
    
    公式：
        x₁ = l₁·sinθ₁,  y₁ = -l₁·cosθ₁
        x₂ = x₁ + l₂·sinθ₂
        y₂ = y₁ - l₂·cosθ₂
    """
    θ1, θ2 = state[0], state[1]
    x1 = l1 * np.sin(θ1)
    y1 = -l1 * np.cos(θ1)
    x2 = x1 + l2 * np.sin(θ2)
    y2 = y1 - l2 * np.cos(θ2)
    return (x1, y1), (x2, y2)

---
## 3️⃣ 数值积分

使用 `scipy.integrate.solve_ivp` 对 ODE 进行数值积分。

**为什么用 `solve_ivp` 而不是手写 RK4？**
- 自适应步长：在需要的时候自动加密，保证 rtol/atol 精度
- 多种方法可选：RK45（默认）、DOP853（高精度）等
- 输出插值：通过 `dense_output=True` 可以在任意时间点取值

In [ ]:
def simulate(initial_state=None, return_energy=False):
    """
    对双摆进行一次完整的数值积分。
    
    返回:
        trajectory : (T, 4) = [θ₁, θ₂, ω₁, ω₂]
        t          : (T,) 时间数组
        (可选) energy : (T,) 总能量
    """
    if initial_state is None:
        initial_state = [THETA1_0, THETA2_0, OMEGA1_0, OMEGA2_0]
    
    n_points = int((T_SPAN[1] - T_SPAN[0]) / DT) + 1
    t_eval = np.linspace(T_SPAN[0], T_SPAN[1], n_points)
    
    sol = solve_ivp(
        derivatives,
        T_SPAN,
        initial_state,
        method=METHOD,
        t_eval=t_eval,
        args=(M1, M2, L1, L2, G),
        rtol=RTOL,
        atol=ATOL,
        dense_output=True,
    )
    
    if not sol.success:
        raise RuntimeError(f"积分失败: {sol.message}")
    
    trajectory = sol.y.T  # (4, T) → (T, 4)
    
    if return_energy:
        energy = np.array([
            compute_energy(s, M1, M2, L1, L2, G)[0]
            for s in trajectory
        ])
        return trajectory, sol.t, energy
    
    return trajectory, sol.t


# ── 跑一条默认轨迹 ──
print("模拟双摆轨迹...")
traj, t = simulate()
print(f"时间点: {len(t)}, 轨迹 shape: {traj.shape}")
print(f"θ₁ 范围: [{traj[:,0].min():.3f}, {traj[:,0].max():.3f}]")
print(f"θ₂ 范围: [{traj[:,1].min():.3f}, {traj[:,1].max():.3f}]")

---
## 4️⃣ 可视化验证

### 4.1 θ(t) 曲线 — 振荡模式

查看角度随时间的变化是否平滑、是否呈现混沌行为。

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)

# 角度
ax = axes[0]
ax.plot(t, traj[:, 0], label=r'$\theta_1$', color='#3498db', lw=1.2)
ax.plot(t, traj[:, 1], label=r'$\theta_2$', color='#e74c3c', lw=1.2)
ax.set_ylabel('Angle [rad]')
ax.legend()
ax.grid(alpha=0.3)
ax.set_title('Angles vs Time')

# 角速度
ax = axes[1]
ax.plot(t, traj[:, 2], label=r'$\omega_1$', color='#2ecc71', lw=1.2)
ax.plot(t, traj[:, 3], label=r'$\omega_2$', color='#f39c12', lw=1.2)
ax.set_xlabel('Time [s]')
ax.set_ylabel('Angular velocity [rad/s]')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 4.2 能量守恒检查

对于保守系统，总能量 $E = T + V$ 应严格守恒。
能量漂移是数值误差的累积——这是评判积分精度的核心指标。

In [ ]:
# 重新积分，同时返回能量
_, _, energy = simulate(return_energy=True)

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)

# 能量组分
ax = axes[0]
# 重建 T 和 V
E_seq = np.zeros(len(t))
T_seq = np.zeros(len(t))
V_seq = np.zeros(len(t))
for i, s in enumerate(traj):
    E_seq[i], T_seq[i], V_seq[i] = compute_energy(s, M1, M2, L1, L2, G)

ax.plot(t, T_seq, label='Kinetic T', color='#2ecc71', lw=1.2)
ax.plot(t, V_seq, label='Potential V', color='#e74c3c', lw=1.2)
ax.plot(t, E_seq, label='Total E = T+V', color='#2c3e50', lw=2)
ax.set_ylabel('Energy [J]')
ax.legend()
ax.grid(alpha=0.3)
ax.set_title('Energy vs Time')

# 能量漂移
ax = axes[1]
delta_E = E_seq - E_seq[0]
ax.plot(t, delta_E, color='#8e44ad', lw=1.5)
ax.set_xlabel('Time [s]')
ax.set_ylabel('$\\Delta E$ [J]')
ax.set_title(f'Energy Drift: max |ΔE| = {np.max(np.abs(delta_E)):.2e}')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"初始能量: {E_seq[0]:.10f}")
print(f"最终能量: {E_seq[-1]:.10f}")
print(f"能量漂移: {E_seq[-1] - E_seq[0]:.2e}")
if abs(E_seq[-1] - E_seq[0]) < 1e-6:
    print("✅ 能量守恒验证通过! 数值积分精度良好。")
else:
    print("⚠️ 能量漂移显著，可能需要提高 rtol/atol。")

---
## 5️⃣ 批量生成轨迹数据集

使用不同的随机初始条件生成多条轨迹。

**为什么需要多条轨迹？**
- **NODE 训练**需要覆盖不同初始条件的轨迹数据，才能学到通用的动力学
- 每条轨迹的初始条件随机采样于 $\theta \in [-\pi, \pi]$, $\omega \in [-3, 3]$

In [ ]:
def generate_dataset(n_trajectories=N_TRAJECTORIES, save_dir='data'):
    """生成多条轨迹并保存为 .npy 文件。"""
    rng = np.random.default_rng(42)
    save_path = Path(save_dir)
    save_path.mkdir(parents=True, exist_ok=True)
    
    info_lines = ["idx,theta1_0,theta2_0,omega1_0,omega2_0"]
    
    for i in range(n_trajectories):
        θ1₀ = rng.uniform(*THETA_RANGE)
        θ2₀ = rng.uniform(*THETA_RANGE)
        ω1₀ = rng.uniform(*OMEGA_RANGE)
        ω2₀ = rng.uniform(*OMEGA_RANGE)
        
        init_state = [θ1₀, θ2₀, ω1₀, ω2₀]
        traj, t = simulate(initial_state=init_state)
        
        # 拼接时间列: (T, 5) = [t, θ₁, θ₂, ω₁, ω₂]
        data = np.column_stack([t, traj])
        np.save(save_path / f"trajectory_{i:05d}.npy", data)
        info_lines.append(f"{i},{θ1₀:.6f},{θ2₀:.6f},{ω1₀:.6f},{ω2₀:.6f}")
    
    with open(save_path / "trajectories_info.csv", "w") as f:
        f.write("\n".join(info_lines))
    
    print(f"✅ 已生成 {n_trajectories} 条轨迹到 {save_path.resolve()}/")


generate_dataset(n_trajectories=10)

---
## 6️⃣ 双摆动画

最直接的物理合理性检查——亲眼看看运动是否合理。

橙色的尾迹是第二个质点的运动轨迹，能直观展示双摆的混沌特征。

In [ ]:
# 预先计算所有帧的质点位置
n_frames = len(traj)
positions = np.zeros((n_frames, 4))
for i, state in enumerate(traj):
    (x1, y1), (x2, y2) = compute_positions(state, L1, L2)
    positions[i] = [x1, y1, x2, y2]

# 坐标范围
margin = 0.3
all_x = positions[:, [0, 2]].flatten()
all_y = positions[:, [1, 3]].flatten()
half_range = max(all_x.max() - all_x.min(), all_y.max() - all_y.min()) / 2
x_center = (all_x.max() + all_x.min()) / 2
y_center = (all_y.max() + all_y.min()) / 2

fig, ax = plt.subplots(figsize=(7, 7))
ax.set_xlim(x_center - half_range, x_center + half_range)
ax.set_ylim(y_center - half_range, y_center + half_range)
ax.set_aspect('equal')
ax.grid(alpha=0.3)
ax.set_title('Double Pendulum Animation')

line, = ax.plot([], [], 'o-', lw=2, markersize=8,
                markerfacecolor='red', markeredgecolor='darkred')
trail_len = min(200, n_frames)
trail, = ax.plot([], [], '-', color='orange', alpha=0.4, lw=1)

def init():
    line.set_data([], [])
    trail.set_data([], [])
    return line, trail

def update(frame):
    x1, y1, x2, y2 = positions[frame]
    line.set_data([0, x1, x2], [0, y1, y2])
    start = max(0, frame - trail_len)
    trail.set_data(positions[start:frame+1, 2],
                   positions[start:frame+1, 3])
    return line, trail

anim = animation.FuncAnimation(
    fig, update, frames=n_frames, init_func=init,
    interval=20, blit=True
)

plt.tight_layout()
plt.show()

# 取消下面注释可以保存动画
# anim.save('data/double_pendulum.gif', writer='pillow', fps=50)

---
## 总结

✅ 完成了双摆模拟 Ver 1：

| 模块 | 状态 | 验证 |
|:----|:----|:----|
| 参数定义 | ✅ | config 化 |
| ODE 方程 | ✅ | `derivatives()` 函数 |
| 数值积分 | ✅ | `solve_ivp` 自适应步长 |
| 能量守恒 | ✅ | $\Delta E < 10^{-6}$ |
| 可视化 | ✅ | 曲线 + 能量 + 动画 |
| 数据集 | ✅ | 多条轨迹 → .npy 文件 |

**下一步：** NODE 训练 — 从轨迹数据反推动力学 $f_\theta(\mathbf{s})$。